In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import pandera as pa
import pandas as pd
from pandera.typing import Series
from dataclasses import dataclass
import phonenumbers
import re

In [3]:
current_dir = Path().resolve().parent
DATA_PATH = current_dir / 'data'

In [4]:
CUSTOMERS_PATH = DATA_PATH / 'customers.csv'
EVENTS_PATH = DATA_PATH / 'events.xml'
ORDERS_PATH = DATA_PATH / 'orders.json'
PAYMENTS_PATH = DATA_PATH / 'payments.csv'
PRODUCTS_PATH = DATA_PATH / 'products.xlsx'

In [5]:
def review_dataset(df):
    print("--- ОБЩАЯ ИНФОРМАЦИЯ ---")
    print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов\n")

    info_df = pd.DataFrame(
        {
            "Тип данных": df.dtypes,
            "Непустые значения": df.count(),
            "Пропуски (NaN)": df.isna().sum(),
            "% Пропусков": (df.isna().sum() / len(df) * 100).round(2),
            "Уникальные": df.nunique(),
        }
    )

    print(info_df)

In [6]:
def base_processing(df):
    df = df.drop_duplicates()
    review_dataset(df)
    return df

# Customers

In [7]:
def standardize_phone_systematic(phone_val):
    if pd.isna(phone_val) or str(phone_val).upper() in ['UNKNOWN', 'NaN', 'N/A', '']:
        return np.nan
    
    phone_str = str(phone_val).strip()
    
    try:
        parsed_number = phonenumbers.parse(phone_str, 'RU')
        if phonenumbers.is_valid_number(parsed_number):
            return phonenumbers.format_number(parsed_number, phonenumbers.PhoneNumberFormat.E164)
    except phonenumbers.NumberParseException:
        pass
    
    return np.nan

def standardize_city_systematic(city_series):
    cleaned = (
        city_series
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'^(п\.|ст\.|г\.|д\.|с\.|клх\.?|поселок|город|станция|деревня|село|к\.)\s*|\s*\([^)]*\)', '', regex=True)
        .str.strip()
        .str.title()
    )
    
    city_mapping = {
        'Питер': 'Санкт-Петербург',
        'Спб': 'Санкт-Петербург',
        'Мск': 'Москва',
        'Набережные Челны': 'Набережные Челны',
        'Клх Печора': 'Печора'
    }
    
    return cleaned.replace(city_mapping)

def standardize_name(name_val):
    if pd.isna(name_val) or str(name_val).upper() in ['UNKNOWN', 'NaN', 'N/A', '']:
        return np.nan
    
    name_str = str(name_val).strip()
    
    # Системное удаление распространенных обращений и мусора в начале строки
    prefixes_to_remove = r'^(г-н|Тов.)\s*'
    name_str = re.sub(prefixes_to_remove, '', name_str, flags=re.IGNORECASE)
    
    # Удаление лишних пробелов и приведение к стандартному виду (Заглавные буквы)
    return " ".join(name_str.split()).title()

def standardize_email_systematic(email_series):
    return (
        email_series
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(['unknown', 'nan', 'null', ''], np.nan)
        .where(lambda x: x.str.match(r'^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$', na=False))
    )


In [8]:
class CustomerSchema(pa.DataFrameModel):
    customer_id: Series[int] = pa.Field(coerce=True, unique=True, nullable=False)
    full_name: Series[str] = pa.Field(coerce=True, nullable=False)
    email: Series[str] = pa.Field(coerce=True, nullable=True)
    phone: Series[str] = pa.Field(coerce=True, nullable=True)
    city: Series[str] = pa.Field(coerce=True, nullable=False)
    created_at: Series[pd.Timestamp] = pa.Field(coerce=True, nullable=False)

    @pa.check("full_name")
    def check_full_name_length(cls, series: Series[str]) -> Series[bool]:
        return series.str.len() >= 3

    @pa.check("email")
    def check_email_format(cls, series: Series[str]) -> Series[bool]:
        return series.str.match(r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$", na=True)

    @pa.check("phone")
    def check_phone_format(cls, series: Series[str]) -> Series[bool]:
        return series.str.match(r"^\+7\d{10}$", na=True)

    @pa.check("city")
    def check_city_length(cls, series: Series[str]) -> Series[bool]:
        return series.str.len() >= 2


/home/admin/Documents/para-files/Projects/git-projects/dit-moscow-test-task/Block 3/.venv/lib64/python3.14/site-packages/pandera/_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [9]:
@dataclass
class ValidationResult:
    is_valid: bool
    clean_df: pd.DataFrame
    quarantine_df: pd.DataFrame
    error_report: pd.DataFrame
    summary: str

def validate_and_quarantine(df: pd.DataFrame, schema: pa.DataFrameModel) -> ValidationResult:
    try:
        validated_df = schema.validate(df, lazy=True)
        
        return ValidationResult(
            is_valid=True,
            clean_df=validated_df,
            quarantine_df=pd.DataFrame(),
            error_report=pd.DataFrame(),
            summary=f"✅ Валидация прошла успешно! Все {len(df)} записей соответствуют формату."
        )
        
    except pa.errors.SchemaErrors as e:
        error_report = e.failure_cases
        
        bad_indices = set()
        for item in error_report['index']:
            if pd.isna(item):
                continue
            if isinstance(item, (list, pd.Series, np.ndarray)):
                bad_indices.update(item)
            else:
                bad_indices.add(item)
        
        bad_indices = list(bad_indices)
        
        quarantine_df = df.loc[bad_indices].copy()
        clean_df = df.drop(index=bad_indices).copy()
        
        unique_errors = error_report[['column', 'check', 'failure_case']].drop_duplicates()
        
        summary = (
            f"❌ Обнаружены ошибки валидации.\n"
            f"📊 Статистика:\n"
            f"   - Всего строк: {len(df)}\n"
            f"   - Чистых строк: {len(clean_df)} ({len(clean_df)/len(df)*100:.1f}%)\n"
            f"   - В карантине: {len(quarantine_df)}\n\n"
            f"📋 Детали ошибок:\n{unique_errors.to_string(index=False)}"
        )
        
        return ValidationResult(
            is_valid=False,
            clean_df=clean_df,
            quarantine_df=quarantine_df,
            error_report=error_report,
            summary=summary
        )

In [10]:
df = pd.read_csv(CUSTOMERS_PATH)
df

,customer_id,full_name,email,phone,city,created_at
0,1,Мишин Емельян Валерьевич,mechislav_37@hotmail.com,UNKNOWN,п. Печора,2024-05-31
1,2,Егоров Тихон Федосеевич,serafim_1986@gmail.com,UNKNOWN,п. Мостовской,2024-07-21
2,3,Зыков Вацлав Алексеевич,fharitonova@gmail.com,9749621470,ст. Луга,2024-06-15
3,4,Савельева Юлия Кузьминична,odintsovgerman@ignatev.biz,NaN,п. Воскресенск,2025-03-28
4,5,София Тимофеевна Михеева,selivan_1979@ao.ru,9206468299,г. Уренгой,2026-01-17
...,...,...,...,...,...,...
710,11,Зыкова Глафира Геннадиевна,silagushchin@mail.ru,NaN,клх Печора,2025-08-20
711,12,Геннадий Матвеевич Цветков,NaN,+79413140753,п. Борзя,2024-09-09
712,13,Мирослав Гертрудович Виноградов,NaN,9119778234,г. Чита,2023-08-26
713,14,г-н Волков Фирс Алексеевич,artem1973@ooo.ru,9320452650,ст. Верхний Тагил,2024-04-12


In [11]:
df['phone'] = df['phone'].apply(standardize_phone_systematic)
df['city'] = standardize_city_systematic(df['city'])
df['full_name'] = df['full_name'].apply(standardize_name)
df['email'] = standardize_email_systematic(df['email'])
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')


In [12]:
df = base_processing(df)

--- ОБЩАЯ ИНФОРМАЦИЯ ---
Размер датасета: 700 строк, 6 столбцов

                 Тип данных  Непустые значения  Пропуски (NaN)  % Пропусков  \
customer_id           int64                700               0         0.00   
full_name               str                700               0         0.00   
email                   str                670              30         4.29   
phone                   str                419             281        40.14   
city                    str                680              20         2.86   
created_at   datetime64[us]                684              16         2.29   

             Уникальные  
customer_id         700  
full_name           691  
email               670  
phone               419  
city                430  
created_at          498  


In [13]:
df

,customer_id,full_name,email,phone,city,created_at
0,1,Мишин Емельян Валерьевич,mechislav_37@hotmail.com,NaN,Печора,2024-05-31
1,2,Егоров Тихон Федосеевич,serafim_1986@gmail.com,NaN,Мостовской,2024-07-21
2,3,Зыков Вацлав Алексеевич,fharitonova@gmail.com,+79749621470,Луга,2024-06-15
3,4,Савельева Юлия Кузьминична,odintsovgerman@ignatev.biz,NaN,Воскресенск,2025-03-28
4,5,София Тимофеевна Михеева,selivan_1979@ao.ru,+79206468299,Уренгой,2026-01-17
...,...,...,...,...,...,...
695,696,Щукина Ульяна Валентиновна,NaN,+79853042013,Волгоград,2025-05-30
696,697,Давыдов Фрол Адамович,evdokim_58@ip.com,+79334783336,Абинск,2024-07-29
697,698,Сократ Владиславович Мельников,mishinignati@general.edu,NaN,Качканар,NaT
698,699,Бобылева Нонна Ниловна,drogov@ooo.com,NaN,Валдай,2024-03-28


In [14]:
result = validate_and_quarantine(df, CustomerSchema)

result.error_report

,schema_context,column,check,check_number,failure_case,index
0,Column,city,not_nullable,None,NaN,17
1,Column,city,not_nullable,None,NaN,98
20,Column,created_at,not_nullable,None,NaT,66
21,Column,created_at,not_nullable,None,NaT,69
22,Column,created_at,not_nullable,None,NaT,138
23,Column,created_at,not_nullable,None,NaT,165
24,Column,created_at,not_nullable,None,NaT,232
25,Column,created_at,not_nullable,None,NaT,325
26,Column,created_at,not_nullable,None,NaT,337
27,Column,created_at,not_nullable,None,NaT,393


In [15]:
result.quarantine_df

,customer_id,full_name,email,phone,city,created_at
640,641,Фаина Болеславовна Пахомова,isakovkliment@zao.com,NaN,NaN,2024-11-22
516,517,Велимир Викентьевич Миронов,sharapovaemilija@hotmail.com,NaN,NaN,2023-07-07
393,394,Маргарита Тимуровна Зыкова,sergeevauljana@ip.ru,+79680202067,Шарья,NaT
138,139,Лора Ильинична Голубева,gusevaelizaveta@rao.net,+79053256086,Верещагино,NaT
521,522,Иванна Эдуардовна Осипова,hohlovemeljan@mail.ru,+79897377234,Балашиха,NaT
652,653,Октябрина Степановна Силина,julian_29@mironov.ru,NaN,Беломорск,NaT
653,654,Щербакова Светлана Рубеновна,noskovvisheslav@yahoo.com,NaN,Темрюк,NaT
17,18,Жданова Екатерина Александровна,sazonovharlampi@zao.edu,+79965953266,NaN,2026-03-05
410,411,Громова Олимпиада Геннадиевна,qjudin@zao.biz,+79121265692,NaN,2024-09-23
671,672,Дарья Георгиевна Петрова,kostinaekaterina@gmail.com,+79066503230,Тимашевск,NaT


In [16]:
df['city'].value_counts()

city
Ангарск                      5
Верхнее Пенжино              4
Нефтекамск                   4
Санкт-Петербург              4
Александровск-Сахалинский    4
                            ..
Зарайск                      1
Арсеньев                     1
Ясный                        1
Жиганск                      1
Валдай                       1
Name: count, Length: 430, dtype: int64

# Events

In [17]:
class EventsSchema(pa.DataFrameModel):
    event_id: Series[int] = pa.Field(
        coerce=True, 
        unique=True, 
        nullable=False, 
        ge=1,
        description="Уникальный идентификатор события"
    )
    
    customer_id: Series[int] = pa.Field(
        coerce=True, 
        nullable=True, 
        ge=1,
        description="ID клиента (может быть null для анонимных пользователей)"
    )
    
    event_type: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        isin=["view", "login", "purchase", "click", "add_to_cart", "logout", "search"],
        description="Тип события из разрешенного списка"
    )
    
    event_timestamp: Series[pd.Timestamp] = pa.Field(
        coerce=True, 
        nullable=False,
        description="Время события, приводится к формату datetime"
    )
    
    product_id: Series[int] = pa.Field(
        coerce=True, 
        nullable=True, 
        ge=1,
        description="ID продукта (может быть null, например, при событии login)"
    )

    @pa.check("event_timestamp")
    def check_timestamp_not_in_future(cls, series: Series[pd.Timestamp]) -> Series[bool]:
        return series <= pd.Timestamp.now() + pd.Timedelta(days=1)

In [18]:
df = pd.read_xml(EVENTS_PATH)

In [19]:
df

,event_id,customer_id,event_type,event_timestamp,product_id
0,1,483.0,view,2025-06-23 18:40:09,281.0
1,2,550.0,login,2025-11-30 00:01:33,20.0
2,3,570.0,login,2025-02-16 21:53:55,588.0
3,4,213.0,view,2025-07-12 08:14:08,408.0
4,5,202.0,purchase,2025-04-03 12:11:22,453.0
...,...,...,...,...,...
1496,1497,32.0,logout,2026-05-14 21:57:34,211.0
1497,1498,62.0,login,broken-date,354.0
1498,1499,609.0,login,2025-06-18 21:07:32,407.0
1499,1500,407.0,click,2024-12-19 22:47:33,141.0


In [ ]:
df['event_id'].isna().sum()

np.int64(0)

In [ ]:
df['event_type'].value_counts()

event_type
view        327
click       321
logout      300
login       283
purchase    270
Name: count, dtype: int64

In [ ]:
df['customer_id'].value_counts()

customer_id
999999.0    65
213.0        6
226.0        6
547.0        6
39.0         6
            ..
41.0         1
36.0         1
197.0        1
32.0         1
62.0         1
Name: count, Length: 644, dtype: int64

In [ ]:
df['event_timestamp'].value_counts()

event_timestamp
broken-date            50
2025-06-23 18:40:09     1
2025-11-30 00:01:33     1
2025-02-16 21:53:55     1
2025-07-12 08:14:08     1
                       ..
2025-08-06 22:10:48     1
2025-10-23 22:34:28     1
2026-05-14 21:57:34     1
2025-06-18 21:07:32     1
2024-12-19 22:47:33     1
Name: count, Length: 1451, dtype: int64

In [ ]:
df['product_id'].isna().sum()

np.int64(1)

In [ ]:
df['customer_id'].isna().sum()

np.int64(1)

In [ ]:
df = df.drop_duplicates()

In [ ]:
result2 = validate_and_quarantine(df, EventsSchema)

print('Обработка events.csv')
print(result2.summary)
print("\n--- ДАННЫЕ В КАРАНТИНЕ ---")
print(result2.quarantine_df)
print("\n--- ЧИСТЫЕ ДАННЫЕ ---")
print(result2.clean_df)

Обработка events.csv
❌ Обнаружены ошибки валидации.
📊 Статистика:
   - Всего строк: 1501
   - Чистых строк: 1450 (96.6%)
   - В карантине: 51

📋 Детали ошибок:
         column                       check                                                         failure_case
       event_id       coerce_dtype('int64')                                                               BAD_ID
event_timestamp                not_nullable                                                                  NaT
       event_id              dtype('int64')                                                               BAD_ID
       event_id greater_than_or_equal_to(1) TypeError("'>=' not supported between instances of 'str' and 'int'")

--- ДАННЫЕ В КАРАНТИНЕ ---
     event_id  customer_id event_type event_timestamp  product_id
512       513          395       view             NaT         205
1411     1412           30      click             NaT         495
643       644          633       view            

In [ ]:
result2.quarantine_df

,event_id,customer_id,event_type,event_timestamp,product_id
512,513,395,view,NaT,205
1411,1412,30,click,NaT,495
643,644,633,view,NaT,108
266,267,706,click,NaT,401
268,269,18,logout,NaT,347
15,16,111,logout,NaT,411
400,401,999999,click,NaT,387
145,146,561,login,NaT,645
1173,1174,558,click,NaT,137
151,152,320,view,NaT,251


# Orders

In [ ]:
class OrdersSchema(pa.DataFrameModel):
    order_id: Series[int] = pa.Field(
        coerce=True, 
        unique=True, 
        nullable=False, 
        ge=1,
        description="Уникальный идентификатор заказа"
    )
    
    customer_id: Series[int] = pa.Field(
        coerce=True, 
        nullable=True,  # Допускаем null, если возможны гостевые заказы
        ge=1,
        description="ID клиента"
    )
    
    product_id: Series[int] = pa.Field(
        coerce=True, 
        nullable=False, 
        ge=1,
        description="ID товара"
    )
    
    quantity: Series[int] = pa.Field(
        coerce=True, 
        nullable=False, 
        gt=0,  # Количество должно быть строго больше 0
        description="Количество единиц товара"
    )
    
    unit_price: Series[float] = pa.Field(
        coerce=True, 
        nullable=False, 
        ge=0.0, 
        le=1_000_000.0,  # Защита от аномальных цен (опечаток вроде 25600000.02)
        description="Цена за единицу товара"
    )
    
    currency: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        isin=["USD", "EUR", "RUB", "GBP", "CNY"], # Белый список валют
        description="Валюта заказа"
    )
    
    order_timestamp: Series[pd.Timestamp] = pa.Field(
        coerce=True, 
        nullable=False,
        description="Время создания заказа"
    )
    
    status: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        isin=["pending", "processing", "completed", "cancelled", "refunded", "shipped"],
        description="Статус заказа"
    )

    @pa.check("order_timestamp")
    def check_timestamp_not_in_future(cls, series: Series[pd.Timestamp]) -> Series[bool]:
        """Заказ не может быть создан в будущем (с допуском в 1 день на часовые пояса)"""
        return series <= pd.Timestamp.now() + pd.Timedelta(days=1)

    @pa.check("order_timestamp")
    def check_timestamp_not_too_old(cls, series: Series[pd.Timestamp]) -> Series[bool]:
        """Защита от аномально старых дат (например, 1970 год из-за ошибок парсинга)"""
        return series >= pd.Timestamp("2020-01-01")


In [ ]:
df = pd.read_json(ORDERS_PATH)
df

,order_id,customer_id,product_id,quantity,unit_price,currency,order_timestamp,status
0,1,380.0,512,3,256.02,USD,2025-99-99,processing
1,2,467.0,117,5,218.67,EUR,2026-02-18 02:28:23,completed
2,3,470.0,391,5,89.70,EUR,2025-01-14 11:22:12,cancelled
3,4,676.0,90,3,354.03,USD,2025-05-30 03:09:57,completed
4,5,92.0,639,4,60.80,RUB,2025-01-22 15:48:14,processing
...,...,...,...,...,...,...,...,...
1215,16,586.0,647,2,491.87,RUB,2026-02-03 08:08:20,cancelled
1216,17,192.0,388,3,104.98,USD,2024-07-27 04:07:00,cancelled
1217,18,676.0,594,2,317.44,RUB,2025-01-05 06:00:33,completed
1218,19,114.0,185,1,79.50,EUR,2025-02-10 05:37:05,completed


In [ ]:
df = base_processing(df)

--- ОБЩАЯ ИНФОРМАЦИЯ ---
Размер датасета: 1200 строк, 8 столбцов

                Тип данных  Непустые значения  Пропуски (NaN)  % Пропусков  \
order_id             int64               1200               0          0.0   
customer_id        float64               1146              54          4.5   
product_id           int64               1200               0          0.0   
quantity             int64               1200               0          0.0   
unit_price         float64               1200               0          0.0   
currency               str               1200               0          0.0   
order_timestamp        str               1200               0          0.0   
status                 str               1200               0          0.0   

                 Уникальные  
order_id               1200  
customer_id             584  
product_id              547  
quantity                  5  
unit_price             1188  
currency                  3  
order_timestamp      

In [ ]:
result3 = validate_and_quarantine(df, OrdersSchema)
print('Обработка events.csv')
print(result3.summary)
print("\n--- ДАННЫЕ В КАРАНТИНЕ ---")
print(result3.quarantine_df)
print("\n--- ЧИСТЫЕ ДАННЫЕ ---")
print(result3.clean_df)

/home/admin/Documents/para-files/Projects/git-projects/dit-moscow-test-task/Block 3/.venv/lib64/python3.14/site-packages/pandera/engines/pandas_engine.py:955: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)


Обработка events.csv
❌ Обнаружены ошибки валидации.
📊 Статистика:
   - Всего строк: 1200
   - Чистых строк: 1166 (97.2%)
   - В карантине: 34

📋 Детали ошибок:
         column                          check                                                               failure_case
order_timestamp coerce_dtype('datetime64[ns]')                                                                 2025-99-99
order_timestamp        dtype('datetime64[ns]')                                                                 2025-99-99
order_timestamp  check_timestamp_not_in_future TypeError("'>=' not supported between instances of 'Timestamp' and 'str'")
order_timestamp    check_timestamp_not_too_old TypeError("'<=' not supported between instances of 'Timestamp' and 'str'")

--- ДАННЫЕ В КАРАНТИНЕ ---
      order_id  customer_id  product_id  quantity  unit_price currency  \
0            1        380.0         512         3      256.02      USD   
1027      1028        117.0         545         5     

In [ ]:
result3.quarantine_df

,order_id,customer_id,product_id,quantity,unit_price,currency,order_timestamp,status
0,1,380.0,512,3,256.02,USD,2025-99-99,processing
1027,1028,117.0,545,5,223.20,EUR,2025-99-99,processing
393,394,618.0,132,4,240.05,EUR,2025-99-99,cancelled
1161,1162,589.0,89,2,423.48,USD,2025-99-99,cancelled
139,140,465.0,394,1,142.45,RUB,2025-99-99,cancelled
269,270,406.0,309,2,108.20,USD,2025-99-99,cancelled
399,400,203.0,166,1,419.34,USD,2025-99-99,cancelled
146,147,244.0,556,3,51.91,USD,2025-99-99,cancelled
1045,1046,677.0,86,3,56.59,RUB,2025-99-99,completed
280,281,415.0,394,3,73.08,RUB,2025-99-99,processing


# Payments

In [ ]:
class PaymentsSchema(pa.DataFrameModel):
    payment_id: Series[int] = pa.Field(
        coerce=True, 
        unique=True, 
        nullable=False, 
        ge=1,
        description="Уникальный идентификатор платежа"
    )
    
    order_id: Series[int] = pa.Field(
        coerce=True, 
        nullable=False, 
        ge=1,
        description="ID заказа, к которому привязан платеж"
    )
    
    payment_method: Series[str] = pa.Field(
        coerce=True, 
        nullable=False, # Платеж БЕЗ способа оплаты - это ошибка данных
        isin=["card", "paypal", "bank_transfer", "crypto", "apple_pay", "google_pay"],
        description="Способ оплаты из разрешенного списка"
    )
    
    amount: Series[float] = pa.Field(
        coerce=True, 
        nullable=False, 
        gt=0.0, # Сумма платежа должна быть строго больше нуля
        le=50_000_000.0, # Защита от аномальных выбросов (опечаток в количестве нулей)
        description="Сумма платежа"
    )
    
    currency: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        isin=["USD", "EUR", "RUB", "GBP", "CNY", "KZT"],
        description="Валюта платежа"
    )
    
    payment_timestamp: Series[pd.Timestamp] = pa.Field(
        coerce=True, 
        nullable=False,
        description="Время совершения платежа"
    )

    # --- КАСТОМНЫЕ ПРОВЕРКИ БИЗНЕС-ЛОГИКИ ---

    @pa.check("payment_timestamp")
    def check_timestamp_not_absurd_future(cls, series: Series[pd.Timestamp]) -> Series[bool]:
        """
        Защита от некорректных дат в будущем. 
        Допускаем максимум +1 год (например, для предзаказов), 
        но это отловит ошибки вроде '2099-01-01' или сдвиги эпохи.
        """
        return series <= pd.Timestamp.now() + pd.Timedelta(days=365)

    @pa.check("payment_timestamp")
    def check_timestamp_not_too_old(cls, series: Series[pd.Timestamp]) -> Series[bool]:
        """Защита от дат до эпохи интернет-платежей (ошибки парсинга в 1970 год)"""
        return series >= pd.Timestamp("2010-01-01")

In [ ]:
df = pd.read_csv(PAYMENTS_PATH,sep='^')
df

,payment_id,order_id,payment_method,amount,currency,payment_timestamp
0,1,778,bank_transfer,error_amount,EUR,2025-04-14 23:23:52
1,2,941,bank_transfer,407.42,USD,2024-12-15 23:01:24
2,3,1173,paypal,1754.7,USD,2025-05-03 14:55:30
3,4,143,card,2372.04,RUB,2026-02-05 00:19:38
4,5,864,card,2366.62,EUR,2025-06-18 21:21:20
...,...,...,...,...,...,...
1007,8,14,bank_transfer,1189.89,RUB,2025-05-05 22:59:37
1008,9,685,NaN,1016.84,EUR,2025-01-18 04:06:47
1009,10,1216,NaN,2574.49,EUR,2026-02-22 05:19:19
1010,11,493,NaN,519.11,RUB,2026-04-05 05:14:09


In [ ]:
df = base_processing(df)

--- ОБЩАЯ ИНФОРМАЦИЯ ---
Размер датасета: 1000 строк, 6 столбцов

                  Тип данных  Непустые значения  Пропуски (NaN)  % Пропусков  \
payment_id             int64               1000               0          0.0   
order_id               int64               1000               0          0.0   
payment_method           str                732             268         26.8   
amount                   str               1000               0          0.0   
currency                 str               1000               0          0.0   
payment_timestamp        str               1000               0          0.0   

                   Уникальные  
payment_id               1000  
order_id                  719  
payment_method              3  
amount                    968  
currency                    3  
payment_timestamp         981  


In [ ]:
result4 = validate_and_quarantine(df, PaymentsSchema)
print('Обработка events.csv')
print(result4.summary)
print("\n--- ДАННЫЕ В КАРАНТИНЕ ---")
print(result4.quarantine_df)
print("\n--- ЧИСТЫЕ ДАННЫЕ ---")
print(result4.clean_df)

Обработка events.csv
❌ Обнаружены ошибки валидации.
📊 Статистика:
   - Всего строк: 1000
   - Чистых строк: 695 (69.5%)
   - В карантине: 305

📋 Детали ошибок:
           column                             check                                                               failure_case
           amount           coerce_dtype('float64')                                                               error_amount
   payment_method                      not_nullable                                                                        NaN
           amount                  dtype('float64')                                                               error_amount
payment_timestamp           dtype('datetime64[ns]')                                                                 13/45/2025
payment_timestamp check_timestamp_not_absurd_future TypeError("'>=' not supported between instances of 'Timestamp' and 'str'")
           amount less_than_or_equal_to(50000000.0)     TypeError("'<=' not su

In [ ]:
quarantine_df = result4.quarantine_df
quarantine_df

,payment_id,order_id,payment_method,amount,currency,payment_timestamp
0,1,778,bank_transfer,error_amount,EUR,2025-04-14 23:23:52
516,517,94,NaN,1099.6,USD,2024-10-07 16:24:37
5,6,1139,NaN,747.97,RUB,2025-02-03 15:11:50
6,7,239,card,error_amount,EUR,2026-01-01 19:45:57
518,519,948,NaN,2530.88,USD,2025-06-22 18:34:15
...,...,...,...,...,...,...
499,500,731,card,error_amount,EUR,2025-06-17 10:28:20
502,503,405,bank_transfer,236.64,USD,13/45/2025
508,509,1127,NaN,1439.29,RUB,2025-06-25 19:35:35
510,511,35,card,943.38,EUR,13/45/2025


In [ ]:
review_dataset(quarantine_df)

--- ОБЩАЯ ИНФОРМАЦИЯ ---
Размер датасета: 305 строк, 6 столбцов

                  Тип данных  Непустые значения  Пропуски (NaN)  % Пропусков  \
payment_id             int64                305               0         0.00   
order_id               int64                305               0         0.00   
payment_method           str                 37             268        87.87   
amount                   str                305               0         0.00   
currency                 str                305               0         0.00   
payment_timestamp        str                305               0         0.00   

                   Уникальные  
payment_id                305  
order_id                  276  
payment_method              3  
amount                    274  
currency                    3  
payment_timestamp         286  


In [ ]:
quarantine_df[quarantine_df['payment_method'].notna() & (quarantine_df['amount'] != 'error_amount')] # ну я пытался сохранить данные, но 13 месяц это слишком

,payment_id,order_id,payment_method,amount,currency,payment_timestamp
67,68,566,paypal,356.27,USD,13/45/2025
693,694,602,paypal,1142.63,EUR,13/45/2025
206,207,952,card,1805.68,USD,13/45/2025
343,344,1091,paypal,738.57,RUB,13/45/2025
423,424,1058,card,733.03,USD,13/45/2025
424,425,1188,card,771.33,EUR,13/45/2025
427,428,690,bank_transfer,2154.76,RUB,13/45/2025
963,964,877,paypal,1934.1,USD,13/45/2025
455,456,1023,paypal,1873.68,RUB,13/45/2025
502,503,405,bank_transfer,236.64,USD,13/45/2025


# Products.xlsx

In [ ]:
class ProductsSchema(pa.DataFrameModel):
    product_id: Series[int] = pa.Field(
        coerce=True, 
        unique=True, 
        nullable=False, 
        ge=1,
        description="Уникальный идентификатор товара"
    )
    
    product_name: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        description="Название товара"
    )
    
    category: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        # Строгий белый список категорий. Если придет "Электроника" вместо "Electronics", оно уйдет в карантин.
        isin=["Electronics", "Clothing", "Home", "Books", "Sports", "Toys", "Food", "Beauty", "Other"],
        description="Категория товара"
    )
    
    price: Series[float] = pa.Field(
        coerce=True, 
        nullable=False, 
        gt=0.0,          # Цена не может быть нулевой или отрицательной
        le=10_000_000.0, # Защита от опечаток (например, 5447000.00 вместо 54.47)
        description="Цена товара"
    )
    
    currency: Series[str] = pa.Field(
        coerce=True, 
        nullable=False,
        # Важно: этот список должен совпадать со списками в таблицах orders и payments для целостности данных
        isin=["USD", "EUR", "RUB", "GBP", "CNY"],
        description="Валюта цены"
    )
    
    is_active: Series[bool] = pa.Field(
        coerce=True, 
        nullable=False,
        description="Флаг активности товара (True/False)"
    )

    # --- КАСТОМНЫЕ ПРОВЕРКИ ---

    @pa.check("product_name")
    def check_name_length(cls, series: Series[str]) -> Series[bool]:
        """Название товара не может быть слишком коротким (защита от мусора вроде 'А', ' ', '-')"""
        return series.str.strip().str.len() >= 2

    @pa.check("product_name")
    def check_name_not_all_caps(cls, series: Series[str]) -> Series[bool]:
        """Опциональная проверка: название не должно быть полностью в верхнем регистре (защита от CAPS LOCK)"""
        # Разрешаем аббревиатуры до 3 символов (например, "TV", "IP"), но запрещаем длинные слова капсом
        return ~series.str.match(r'^[A-ZА-ЯЁ\s]{4,}$')

In [ ]:
df = pd.read_excel(PRODUCTS_PATH)
df

,product_id,product_name,category,price,currency,is_active
0,1,Господь,Home,54.47,USD,False
1,2,Набор,Books,114.79,EUR,True
2,3,Вздрагивать,Electronics,49.98,RUB,False
3,4,Фонарик,Electronics,392.93,RUB,False
4,5,Фонарик,Clothing,798.13,EUR,True
...,...,...,...,...,...,...
605,6,Холодно,Sports,572.88,USD,True
606,7,Порядок,Clothing,1457.07,EUR,True
607,8,Академик,Clothing,1612.32,USD,True
608,9,Куча,Sports,291.56,EUR,True


In [ ]:
df = base_processing(df)

--- ОБЩАЯ ИНФОРМАЦИЯ ---
Размер датасета: 600 строк, 6 столбцов

             Тип данных  Непустые значения  Пропуски (NaN)  % Пропусков  \
product_id        int64                600               0         0.00   
product_name        str                600               0         0.00   
category            str                600               0         0.00   
price           float64                581              19         3.17   
currency            str                600               0         0.00   
is_active          bool                600               0         0.00   

              Уникальные  
product_id           600  
product_name         357  
category               5  
price                581  
currency               3  
is_active              2  


In [ ]:
result5 = validate_and_quarantine(df, ProductsSchema)
print('Обработка events.csv')
print(result5.summary)
print("\n--- ДАННЫЕ В КАРАНТИНЕ ---")
print(result5.quarantine_df)
print("\n--- ЧИСТЫЕ ДАННЫЕ ---")
print(result5.clean_df)

Обработка events.csv
❌ Обнаружены ошибки валидации.
📊 Статистика:
   - Всего строк: 600
   - Чистых строк: 580 (96.7%)
   - В карантине: 20

📋 Детали ошибок:
      column             check failure_case
product_name check_name_length            О
       price      not_nullable          NaN

--- ДАННЫЕ В КАРАНТИНЕ ---
     product_id product_name     category   price currency  is_active
391         392    Сохранять         Home     NaN      EUR      False
135         136     Потрясти       Sports     NaN      EUR      False
394         395       Райком     Clothing     NaN      RUB       True
525         526        Точно  Electronics     NaN      USD       True
404         405      Холодно         Home     NaN      EUR      False
23           24   Неожиданно         Home     NaN      EUR      False
283         284       Стакан        Books     NaN      USD      False
413         414       Анализ         Home     NaN      USD      False
312         313    Факультет        Books     NaN   

In [ ]:
result5.quarantine_df

,product_id,product_name,category,price,currency,is_active
391,392,Сохранять,Home,NaN,EUR,False
135,136,Потрясти,Sports,NaN,EUR,False
394,395,Райком,Clothing,NaN,RUB,True
525,526,Точно,Electronics,NaN,USD,True
404,405,Холодно,Home,NaN,EUR,False
23,24,Неожиданно,Home,NaN,EUR,False
283,284,Стакан,Books,NaN,USD,False
413,414,Анализ,Home,NaN,USD,False
312,313,Факультет,Books,NaN,USD,False
191,192,Порядок,Sports,NaN,RUB,True
